cd ~/BiConVarNet/

cat fasta_files/*.fasta > merged.fasta

grep -c "^>" merged.fasta

mmseqs createdb merged.fasta merged_db

>>이게 제대로 

mmseqs search merged_db uniref90_mmseqs merged_result merged_tmp --threads 20 -e 0.001 --max-seqs 5000

mmseqs search merged_db uniref90_mmseqs merged_result merged_tmp --threads 36 -e 0.001 --max-seqs 500

mmseqs align merged_db uniref90_mmseqs merged_result merged_aln --threads 20 - > --add-self-matches가 오류 일으킨다는 말 찾음...

mmseqs align merged_db uniref90_mmseqs merged_result merged_aln --threads 36 --add-self-matches -> 서버용

mkdir msa_output

mmseqs convertmsa merged_db uniref90_mmseqs merged_aln msa_output --msa-format-mode 2 --format-output a3m


>> 아래는 m8 만들기

mmseqs convertalis merged_db uniref90_mmseqs merged_result mmseqs_results.m8 --format-output "query,target,evalue,alnlen,pident,qstart,qend,tstart,tend,qlen,tlen,qseq,tseq"

서버
mmseqs search merged_db uniref90_mmseqs merged_result_2 merged_tmp --threads 36 -e 0.001 --max-seqs 500

mmseqs align merged_db uniref90_mmseqs merged_result_2 merged_aln --threads 36 --add-self-matches -> 필요가 없었네... 머쓱


mmseqs result2msa merged_db uniref90_mmseqs merged_result_2 merged_msa_db --msa-format-mode 2 --threads 36 ->2는 파씽이 어려울듯

mmseqs result2msa merged_db uniref90_mmseqs merged_result_2 merged_msa_a3m \
  --msa-format-mode 5 \
  --threads 36 \
  --max-seq-id 0.95 \
  --qid 0.3 \
  --cov 0.3 \
  --filter-msa 1 \
  --filter-min-enable 100 \
  --diff 500

mmseqs convertmsa merged_msa_db msa_output --msa-format-mode 2 --format-output a3m

PC

mmseqs result2msa merged_db uniref90_mmseqs merged_result merged_msa_db --msa-format-mode 2 --threads 20

mmseqs result2msa merged_db uniref90_mmseqs merged_result merged_msa_a3m \
  --msa-format-mode 5 \
  --threads 20 \
  --max-seq-id 0.95 \
  --qid 0.3 \
  --cov 0.3 \
  --filter-msa 1 \
  --filter-min-enable 100 \
  --diff 1000


밑에서부터는 랩소디 코드 날린거...

In [ ]:
import json

json_path = r"C:\Users\Kunny\Research\Dataset\Missense Variant dataset\UniProtID_to_seq.json"

with open(json_path, "r") as f:
    id_to_seq = json.load(f)

query_lengths = {uid: len(seq) for uid, seq in id_to_seq.items()}


with open(r"C:\Users\Kunny\Research\Dataset\Missense Variant dataset\query_lengths.json", "w") as f:
    json.dump(id_to_seq, f)

In [ ]:
import os
from collections import defaultdict
from tqdm import tqdm  

def parse_mmseqs_msa(m8_file, output_dir, query_lengths=None, min_coverage=30, max_evalue=0.001):
    print(f"Parsing MMseqs2 M8 results from {m8_file} to create individual .seqmsa files in {output_dir}...")
    os.makedirs(output_dir, exist_ok=True)

    msa_dict = defaultdict(list)

    with open(m8_file, 'r') as f:
        total_lines = sum(1 for _ in f)

    with open(m8_file, 'r') as f, tqdm(total=total_lines, desc="Processing alignments") as pbar:
        for line in f:
            parts = line.strip().split('\t')
            pbar.update(1)

            if len(parts) < 13:
                continue
            query_id = parts[0]
            evalue = float(parts[2])
            qstart, qend = int(parts[5]), int(parts[6])
            qseq, tseq = parts[11], parts[12]

            qlen = query_lengths.get(query_id, int(parts[9])) if query_lengths else int(parts[9])
            coverage = (qend - qstart + 1) / qlen * 100
            if coverage < min_coverage or evalue > max_evalue:
                continue

            pre_pad = '-' * (qstart - 1)
            post_pad = '-' * (qlen - qend)
            aligned_seq = pre_pad + tseq + post_pad
            aligned_seq = aligned_seq[:qlen].ljust(qlen, '-')

            msa_dict[query_id].append(aligned_seq)

    print("Writing individual .seqmsa files...")
    for query_id, msa_lines in tqdm(msa_dict.items(), desc="Saving MSAs"):
        if msa_lines:
            msa_path = os.path.join(output_dir, f"{query_id}.seqmsa")
            with open(msa_path, 'w', newline='\n') as out:
                for line in msa_lines:
                    out.write(line + '\n')
        else:
            print(f"[{query_id}] No valid alignments passed filters. Skipping.")

    print("All MSA files generated.")


In [13]:
msa_dir = r"E:\CAGI_data\MSA_folder"

In [ ]:
parse_mmseqs_msa(
    m8_file=r"E:\CAGI_data\mmseqs_results.m8",
    output_dir=msa_dir,
    query_lengths=query_lengths
)

Parsing MMseqs2 M8 results from E:\CAGI_data\mmseqs_results.m8 to create individual .seqmsa files in E:\CAGI_data\MSA_folder...


Processing alignments: 100%|██████████| 62702052/62702052 [07:30<00:00, 139228.06it/s]


Writing individual .seqmsa files...


Saving MSAs: 100%|██████████| 12094/12094 [10:10<00:00, 19.82it/s]


✅ All MSA files generated.


In [20]:
import os
import json
from collections import Counter
from tqdm import tqdm

def precompute_and_save_all_frequencies(msa_dir, output_dir):
    print(f"Scanning MSA files in {msa_dir}...")
    os.makedirs(output_dir, exist_ok=True)

    msa_files = [f for f in os.listdir(msa_dir) if f.endswith(".seqmsa")]

    for msa_file in tqdm(msa_files, desc="Processing .seqmsa files"):
        msa_path = os.path.join(msa_dir, msa_file)
        try:
            with open(msa_path, 'r') as f:
                sequences = [line.strip() for line in f if line.strip()]
        except Exception as e:
            print(f"[ERROR] Could not read {msa_path}: {e}")
            continue

        if len(sequences) < 3:
            tqdm.write(f"[SKIP] {msa_file}: too few sequences (<3)")
            continue

        seq_len = len(sequences[0])
        position_freqs = [Counter() for _ in range(seq_len)]
        overall_freqs = Counter()

        for seq in sequences:
            for i, aa in enumerate(seq):
                position_freqs[i][aa] += 1
                overall_freqs[aa] += 1

        total_seqs = len(sequences)
        total_aa_counts = sum(overall_freqs.values())

        overall_freqs = {aa: count / total_aa_counts for aa, count in overall_freqs.items()}
        position_freqs = [
            {aa: count / total_seqs for aa, count in pos_freq.items()}
            for pos_freq in position_freqs
        ]

        output_data = {
            "overall_freqs": overall_freqs,
            "position_freqs": position_freqs
        }

        json_name = msa_file.replace(".seqmsa", ".seqmsa.json")
        json_path = os.path.join(output_dir, json_name)

        try:
            with open(json_path, 'w') as jf:
                json.dump(output_data, jf)
        except Exception as e:
            tqdm.write(f"[ERROR] Could not write {json_name}: {e}")
            continue


In [ ]:
precompute_and_save_all_frequencies(
    msa_dir="E:/CAGI_data/MSA_folder",
    output_dir="E:/CAGI_data/PSIC_folder"
) #느려서 generate_json_from_seqmsa.py로 진행

Scanning MSA files in E:/CAGI_data/MSA_folder...


Processing .seqmsa files:   0%|          | 10/12094 [00:15<5:02:34,  1.50s/it]


KeyboardInterrupt: 

In [ ]:
from old_scripts.generate_json_from_seqmsa import precompute_and_save_all_frequencies_parallel

precompute_and_save_all_frequencies_parallel(
        msa_dir="E:/CAGI_data/MSA_folder",
        output_dir="E:/CAGI_data/PSIC_folder",
        max_workers=12
    )

Scanning MSA files in E:/CAGI_data/MSA_folder...


Parallel processing: 100%|██████████| 12094/12094 [3:40:02<00:00,  1.09s/it]  


In [36]:
import os
import json
import math
import prody as pr
from tqdm import tqdm

# Define amino acid groups including gap '-' as a subgroup
amino_acid_groups1 = {
    'hydrophobic': {'A', 'V', 'I', 'L', 'M', 'F', 'W', 'P'},
    'polar': {'S', 'T', 'Y', 'N', 'Q', 'C', 'G'},
    'positive': {'K', 'R', 'H'},
    'negative': {'D', 'E'},
    'other': {'B', 'J', 'Z', 'X'},
    'gap': {'-'}
}

amino_acid_groups2 = {
    'smallest': {'G', 'A', 'S'},
    'small': {'T', 'C', 'P', 'D', 'N', 'V'},
    'medium': {'I', 'E', 'Q', 'L', 'H', 'M'},
    'large': {'F', 'Y', 'W', 'R', 'K'},
    'other': {'B', 'J', 'Z', 'X'},
    'gap': {'-'}
}

aa_codes = ['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']

def get_amino_acid_group(amino_acid, groups):
    for group, amino_acids in groups.items():
        if amino_acid in amino_acids:
            return group
    return 'unknown'

def calculate_psic(precomputed_folder, msa_filename, position, amino_acid, groups):
    precomputed_path = os.path.join(precomputed_folder, os.path.basename(msa_filename).replace('.pdb', '') + '.seqmsa.json')
    
    try:
        with open(precomputed_path, 'r') as f:
            data = json.load(f)
    except Exception as e:
        print(f"Error reading precomputed JSON file: {e}")
        return float('NaN'), float('NaN'), 'unknown'
    
    overall_freqs = data.get('overall_freqs', {})
    position_freqs = data.get('position_freqs', [])

    if not position_freqs or position < 1 or position > len(position_freqs):
        print("Position out of range or no position frequencies available")
        return float('NaN'), float('NaN'), 'unknown'
    
    position -= 1
    column_freqs = position_freqs[position]
    
    total_seqs = sum(column_freqs.values())
    observed_freq = column_freqs.get(amino_acid, 0) / total_seqs if total_seqs > 0 else 0
    
    total_aa_counts = sum(overall_freqs.values())
    expected_freq = overall_freqs.get(amino_acid, 0) / total_aa_counts if total_aa_counts > 0 else 0
    
    psic_score = float('-inf') if observed_freq == 0 or expected_freq == 0 else math.log(observed_freq / expected_freq)
    
    amino_acid_group = get_amino_acid_group(amino_acid, groups)
    if amino_acid_group == 'unknown':
        print(f"Unknown amino acid: {amino_acid}")
        return float('NaN'), float('NaN'), 'unknown'
    
    group_freq = sum(column_freqs.get(aa, 0) for aa in groups[amino_acid_group]) / total_seqs if total_seqs > 0 else 0
    group_expected_freq = sum(overall_freqs.get(aa, 0) for aa in groups[amino_acid_group]) / total_aa_counts if total_aa_counts > 0 else 0
    group_psic_score = float('-inf') if group_freq == 0 or group_expected_freq == 0 else math.log(group_freq / group_expected_freq)

    return psic_score, group_psic_score, amino_acid_group

def generate_saturation_mutagenesis_features_fasta(uniprot_id, sequence, precomputed_folder, output_file):
    print(f"[{uniprot_id}] Generating saturation mutagenesis features...")

    output_data = []
    for i, wt_residue in enumerate(sequence):
        pos = i + 1  # 1-based index
        if wt_residue not in aa_codes:
            continue  # Skip unknowns
        for mut_residue in aa_codes:
            wt_psic, wt_group_psic, _ = calculate_psic(precomputed_folder, uniprot_id, pos, wt_residue, amino_acid_groups1)
            wt_size_psic, _, _ = calculate_psic(precomputed_folder, uniprot_id, pos, wt_residue, amino_acid_groups2)
            mut_psic, mut_group_psic, _ = calculate_psic(precomputed_folder, uniprot_id, pos, mut_residue, amino_acid_groups1)
            mut_size_psic, _, _ = calculate_psic(precomputed_folder, uniprot_id, pos, mut_residue, amino_acid_groups2)
            wt_mut_psic = wt_psic - mut_psic
            wt_mut_group_psic = wt_group_psic - mut_group_psic
            wt_mut_size_psic = wt_size_psic - mut_size_psic
            output_data.append([
                uniprot_id,
                pos,
                wt_residue,
                mut_residue,
                f"{wt_psic:.3f}",
                f"{wt_group_psic:.3f}",
                f"{wt_size_psic:.3f}",
                f"{mut_psic:.3f}",
                f"{mut_group_psic:.3f}",
                f"{mut_size_psic:.3f}",
                f"{wt_mut_psic:.3f}",
                f"{wt_mut_group_psic:.3f}",
                f"{wt_mut_size_psic:.3f}"
            ])

    with open(output_file, "w") as f:
        f.write("PDB_File\tResidue_Number\tWT_Residue\tMutation\tWT_Seq_PSIC\tWT_PhyChem_Group_Seq_PSIC\tWT_Size_Group_Seq_PSIC\tMut_Seq_PSIC\tMut_PhyChem_Group_Seq_PSIC\tMut_Size_Group_Seq_PSIC\tWT-Mut_Seq_PSIC\tWT-Mut_PhyChem_Group_Seq_PSIC\tWT-Mut_Size_Group_Seq_PSIC\n")
        for row in output_data:
            f.write("\t".join(map(str, row)) + "\n")

def run_batch_saturation_feature_gen(json_seq_path, precomputed_folder, output_dir):
    with open(json_seq_path, 'r') as f:
        uniprot_dict = json.load(f)

    os.makedirs(output_dir, exist_ok=True)

    json_files = [f for f in os.listdir(precomputed_folder) if f.endswith(".seqmsa.json")]
    for json_file in tqdm(json_files, desc="Generating PSIC features"):
        uniprot_id = json_file.replace(".seqmsa.json", "")
        if uniprot_id not in uniprot_dict:
            print(f"[WARN] {uniprot_id} not found in sequence dict.")
            continue
        sequence = uniprot_dict[uniprot_id]
        output_file = os.path.join(output_dir, f"{uniprot_id}_saturation.tsv")
        generate_saturation_mutagenesis_features_fasta(uniprot_id, sequence, precomputed_folder, output_file)



In [ ]:
from old_scripts.generate_saturation_features import run_batch_saturation_feature_gen_parallel

run_batch_saturation_feature_gen_parallel(
    json_seq_path=r"C:\Users\Kunny\Research\Dataset\Missense Variant dataset\UniProtID_to_seq.json",
    precomputed_folder="E:/CAGI_data/PSIC_folder",
    output_dir="E:/CAGI_data/saturation_features",
    num_workers=6  # CPU 코어 수에 맞게 조정
)

Launching 12094 jobs with 6 workers...


  0%|          | 2/12094 [01:04<108:09:24, 32.20s/it]
